# TopicGPT: LLM-based Topic Modeling

Implementation of TopicGPT (NAACL 2024) with modifications:
- Uses **local LM Studio API** (`mistralai/ministral-3-3b`)
- 3-stage pipeline: **Generation → Refinement → Assignment**
- Evaluates with **Coherence (C_v)**, **IRBO Diversity**, and **Topic Quality**
- Checkpointing support for resumable execution

In [1]:
import os
import re
import json
import time
import random
import pickle
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Configuration

In [2]:
# ─── Subjects & Paths ───
LIST_SUBJECT = ["cs", "math", "physics"]
VERSION = "v1"

BASE_DIR = Path("../../../../data/preprocess")
RESULT_DIR = Path("../../../../results/topicGpt/modeling")
CHECKPOINT_DIR = Path("../../../../models/topicGpt")

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)
    (CHECKPOINT_DIR / subject).mkdir(parents=True, exist_ok=True)

# ─── LM Studio API Config ───
LLM_API_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "mistralai/ministral-3-3b"
LLM_TEMPERATURE = 0.2
LLM_MAX_TOKENS = 39000

# ─── TopicGPT Pipeline Config ───
GENERATION_SAMPLE_SIZE = 500      # docs sampled per generation batch
GENERATION_BATCH_SIZE = 5         # docs per LLM call in generation
GENERATION_MAX_BATCHES = 100      # max LLM calls for generation
ASSIGNMENT_BATCH_SIZE = 5       # docs per LLM call in assignment

# ─── Coherence Config ───
TOP_N_WORDS = 10
RBO_P = 0.9

print(f"Subjects: {LIST_SUBJECT}")
print(f"LLM: {LLM_MODEL} @ {LLM_API_URL}")
print(f"Results: {RESULT_DIR.resolve()}")

Subjects: ['cs', 'math', 'physics']
LLM: mistralai/ministral-3-3b @ http://localhost:1234/v1/chat/completions
Results: /home/nedo/Kuliah/TA/Program/results/topicGpt/modeling


## LLM API Helper

In [3]:
def call_llm(system_prompt: str, user_prompt: str, max_retries: int = 3) -> str:
    """Call LM Studio API with retry logic."""
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": LLM_TEMPERATURE,
        "max_tokens": LLM_MAX_TOKENS,
    }
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                LLM_API_URL,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
            
            # Handle both OpenAI-style and LM Studio response formats
            if "choices" in data:
                return data["choices"][0]["message"]["content"].strip()
            elif "content" in data:
                return data["content"].strip()
            elif "output" in data:
                return data["output"].strip()
            else:
                return str(data)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt+1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"  LLM call failed after {max_retries} attempts: {e}")
                return ""

# Test connection
test_resp = call_llm("You are a helpful assistant.", "Say 'OK' if you can read this.")
print(f"LLM connection test: {test_resp[:100]}")

  Retry 1/3 after 1s: HTTPConnectionPool(host='localhost', port=1234): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f546cf8a7b0>: Failed to establish a new connection: [Errno 111] Connection refused'))
  Retry 2/3 after 2s: HTTPConnectionPool(host='localhost', port=1234): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f546d1cfed0>: Failed to establish a new connection: [Errno 111] Connection refused'))
  LLM call failed after 3 attempts: HTTPConnectionPool(host='localhost', port=1234): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f546cff07d0>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM connection test: 


## Checkpoint Utilities

In [4]:
def save_checkpoint(data, name: str, subject: str):
    """Save checkpoint to disk."""
    path = CHECKPOINT_DIR / subject / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint saved: {path}")

def load_checkpoint(name: str, subject: str):
    """Load checkpoint from disk, return None if not found."""
    path = CHECKPOINT_DIR / subject / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint loaded: {path}")
        return data
    return None

## Data Loading

In [5]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load preprocessed dataset."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    return df

all_data = {}
for subject in LIST_SUBJECT:
    df = load_dataset(subject)
    all_data[subject] = df
    print(f"{subject}: {len(df):,} documents loaded")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Sample text: {str(df['text'].iloc[0])[:120]}...")

cs: 165,756 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: fault detection using immune based systems and formal language algorithms this paper describes two approaches for fault ...
math: 157,085 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: supersymmetry and homotopy the homotopical information hidden in a supersymmetric structure is revealed by considering d...
physics: 146,311 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: critical dynamics of gelation shear relaxation and dynamic density fluctuations are studied within a rouse model general...


---
## Stage 1: Topic Generation

Iteratively prompt the LLM with batches of documents. For each batch, the LLM sees existing topics and either identifies existing topics or proposes new ones.

In [6]:
GENERATION_SYSTEM_PROMPT = """You are an expert topic analyst. Your task is to identify generalizable topics from academic paper abstracts.

Rules:
- Each topic must be GENERAL and BROAD enough to cover multiple papers
- Topic labels must be concise (2-5 words)
- Each topic needs a short description
- Do NOT create overly specific topics tied to a single paper
- Each topic should represent a SINGLE concept, not a combination
- You MUST output your response strictly as a valid JSON object."""

GENERATION_USER_TEMPLATE = """Below are the current topics discovered so far:

[Current Topics]
{topics}

[Document]
{document}

[Instructions]
1. Read the document above.
2. Determine if it fits an existing topic from the list.
3. Respond ONLY with a valid JSON object. Do not include markdown formatting, explanations, or extra text.

If an existing topic fits well, use this JSON schema:
{{
    "is_new": false,
    "topic_id": 42
}}

If no existing topic fits and you must create a new one, use this JSON schema:
{{
    "is_new": true,
    "label": "Your Concise Label",
    "description": "A short, generalized description of the concept"
}}

Your response:"""

In [7]:
import time
def format_topics_for_prompt(topics: dict) -> str:
    if not topics:
        return "No topics exist yet. You must create a [NEW] topic."
    
    formatted_lines = []
    for tid, t_info in topics.items():
        label = t_info.get("label", "Unknown Label")
        desc = t_info.get("description", "No description provided.")
        # Formatting exactly so the LLM knows the ID to return
        formatted_lines.append(f"ID {tid} | {label}: {desc}")
        
    return "\n".join(formatted_lines)

def stratified_sample_by_year(df, min_per_year=100, pct=0.05):
    years = pd.to_datetime(df['submitted_date']).dt.year
    
    sampled_indices = []
    
    # 2. Group the original dataframe indices by these years
    for year, group_indices in df.groupby(years).groups.items():
        n_total = len(group_indices)
        
        # Determine sample size: max of pct or min_per_year
        n_pct = int(n_total * pct)
        n_to_sample = max(min_per_year, n_pct)
        
        # Ensure we don't try to sample more than exists in that year
        n_to_sample = min(n_total, n_to_sample)
        
        # Randomly sample from the index list of this group
        # Using numpy for a quick shuffle/choice
        year_sample = df.loc[group_indices].sample(n=n_to_sample, random_state=42).index.tolist()
        sampled_indices.extend(year_sample)
        
    print(f"  Sampling complete. Total docs for topic generation: {len(sampled_indices)}")
    return sampled_indices

def parse_generation_response(response: str, topics: dict):
    """
    Parses a JSON response from the LLM.
    Returns (assigned_topic_id, new_topic_dict).
    """
    response = response.strip()
    
    # Strip markdown code block formatting if the LLM includes it
    response = re.sub(r"^```json\s*", "", response, flags=re.IGNORECASE)
    response = re.sub(r"\s*```$", "", response)
    
    try:
        data = json.loads(response)
        
        # Case 1: Existing Topic
        if data.get("is_new") is False:
            tid = data.get("topic_id")
            if isinstance(tid, int) and tid in topics:
                return tid, None
                
        # Case 2: New Topic
        elif data.get("is_new") is True:
            label = data.get("label", "").strip()
            desc = data.get("description", "").strip()
            if label and desc:
                return None, {"label": label, "description": desc}
                
    except json.JSONDecodeError:
        print(f"  [Warning] Failed to parse JSON: {response}")
        # Optionally, you could log this or handle retries here
        
    # Fallback if the format was wrong or topic ID didn't exist
    return None, None

def generate_topics(df, subject):
    """Stage 1: one-doc-at-a-time generation with doc->topic mapping."""
    checkpoint = load_checkpoint("generation", subject)
    if checkpoint is not None and "doc_map" in checkpoint:
        topics    = checkpoint["topics"]
        doc_map   = checkpoint["doc_map"]    # {doc_idx: topic_id}
        processed = checkpoint["processed"]  # set of doc_idx already done
        print(f"  Resumed: {len(topics)} topics, {len(processed)} docs processed")
    else:
        topics, doc_map, processed = {}, {}, set()

    sampled_indices = stratified_sample_by_year(df, min_per_year=100, pct=0.05)
    texts = df["text"].fillna("").tolist()
    next_id = max(topics.keys(), default=0) + 1

    todo = [i for i in sampled_indices if i not in processed]
    if not todo:
        print("  All docs already processed.")
        return topics, doc_map

    pbar = tqdm(todo, desc="Generating topics")
    for doc_idx in pbar:
        text = texts[doc_idx][:38000]
        topics_str = format_topics_for_prompt(topics)
        user_prompt = GENERATION_USER_TEMPLATE.format(
            topics=topics_str, document=text
        )
        response = call_llm(GENERATION_SYSTEM_PROMPT, user_prompt)
        
        tid, new_topic = parse_generation_response(response, topics)
        

        if new_topic:
            topics[next_id] = new_topic
            tid = next_id
            next_id += 1

        if tid is not None:
            doc_map[doc_idx] = tid

        processed.add(doc_idx)
        pbar.set_postfix({"topics": len(topics), "mapped": len(doc_map)})

        if len(processed) % 50 == 0:
            save_checkpoint(
                {"topics": topics, "doc_map": doc_map, "processed": processed},
                "generation", subject
            )

    save_checkpoint(
        {"topics": topics, "doc_map": doc_map, "processed": processed},
        "generation", subject
    )
    print(f"  Generated {len(topics)} topics, mapped {len(doc_map)} docs")
    return topics, doc_map

In [8]:
all_topics  = {}
all_doc_maps = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC GENERATION: {subject.upper()}")
    print(f"{'='*60}")
    topics, doc_map = generate_topics(all_data[subject], subject)
    all_topics[subject]   = topics
    all_doc_maps[subject] = doc_map
    print(f"\n  Topics for {subject}:")
    for tid, t in sorted(topics.items()):
        print(f"    [{tid}] {t['label']}: {t['description']}")


TOPIC GENERATION: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/generation.pkl
  Resumed: 621 topics, 8906 docs processed
  Sampling complete. Total docs for topic generation: 8906
  All docs already processed.

  Topics for cs:
    [1] Web Text Database Classification: Automated methods for categorizing and probing search-only text databases via query-based classification techniques
    [2] AI Story Understanding: Exploration of computational methods for interpreting narrative structures and semantic coherence in textual data, including historical shifts and agent-based approaches
    [3] Parallel Debugging Systems: Techniques and tools for diagnosing errors in concurrent/parallel computing environments, focusing on visualization, nondeterminism handling, and distributed debugging mechanisms
    [4] Noninvertibility Theory: Exploration of mathematical and cryptographic properties defining invertibility limits in functions, particularly focusing on novel definitions (e.g., st

In [9]:
from collections import Counter

def show_significant_topics(all_topics, all_doc_maps, min_docs=2):
    """
    Displays statistics for topics with more than 'min_docs' assignments.
    """
    for subject, topics in all_topics.items():
        print(f"\n{'='*65}")
        print(f"SIGNIFICANT TOPICS (Docs > {min_docs-1}): {subject.upper()}")
        print(f"{'='*65}")
        
        # Get counts for the current subject
        doc_map = all_doc_maps.get(subject, {})
        counts = Counter(doc_map.values())
        
        # Sort topics by frequency (highest first)
        sorted_tids = sorted(counts.items(), key=lambda item: item[1], reverse=True)
        
        total_mapped = sum(counts.values())
        print(f"{'ID':<5} | {'Count':<8} | {'Share %':<8} | {'Topic Label'}")
        print("-" * 65)
        
        found_any = False
        for tid, count in sorted_tids:
            if count >= min_docs:
                label = topics.get(tid, {}).get('label', 'Unknown')
                share = (count / total_mapped) * 100 if total_mapped > 0 else 0
                
                print(f"{tid:<5} | {count:<8} | {share:>6.1f}%  | {label}")
                found_any = True
        
        if not found_any:
            print(f"No topics found with more than {min_docs-1} documents.")
            
        print(f"\nUnique topics in {subject}: {len(topics)}")
        print(f"Significant topics: {len([c for c in counts.values() if c >= min_docs])}")

# Execution
show_significant_topics(all_topics, all_doc_maps, min_docs=10)


SIGNIFICANT TOPICS (Docs > 9): CS
ID    | Count    | Share %  | Topic Label
-----------------------------------------------------------------
14    | 1085     |   12.2%  | Repository Management Systems
21    | 498      |    5.6%  | Reproducibility Pitfalls
140   | 496      |    5.6%  | Periodic Numeration Systems
23    | 487      |    5.5%  | Dynamic Market Optimization
153   | 469      |    5.3%  | Time-Series Pattern Discovery
16    | 439      |    4.9%  | Polysemantic Feature Modeling
143   | 292      |    3.3%  | Context-Dependent Classification
15    | 259      |    2.9%  | Recursive Definition Systems
25    | 222      |    2.5%  | Variable Word Rate Modeling
20    | 220      |    2.5%  | Foundations of Mathematical Randomness
73    | 211      |    2.4%  | Regulatory Agency Independence
29    | 205      |    2.3%  | Multi-Channel Rate Distortion Optimization
103   | 179      |    2.0%  | Conservative Parallel Simulation Scalability
173   | 176      |    2.0%  | Epistemic Blame Th

---
## Stage 2: Topic Refinement

Merge near-duplicate or overlapping topics using the LLM.

In [10]:
REFINEMENT_SYSTEM_PROMPT = """You are an expert at organizing topic taxonomies. Your task is to merge topics that are near-duplicates, synonyms, or heavily overlapping.

Rules:
- Only merge topics that are truly redundant or nearly identical
- Keep the most general and descriptive label
- Return the merge operations in the specified format
- If no merges are needed, return "None" """

REFINEMENT_USER_TEMPLATE = """
You are given a list of topics. Identify groups that should be merged because they are near-duplicates or heavily overlapping.

[Topic List]
{topics}

[Task]
Detect topics representing the same concept and propose merges.

[Output Format]
Return ONLY valid JSON.

{{
  "merges": [
    {{
      "merge_ids": [id1, id2, ...],
      "kept_id": id,
      "label": "New merged topic label",
      "description": "Short description of the merged topic",
      "confidence": 0.0
    }}
  ]
}}

Rules:
- "merge_ids" must contain all topic IDs being merged
- "kept_id" must be one of the IDs inside merge_ids
- "confidence" must be between 0.0 and 1.0

If no merges are needed return:

{{
  "merges": []
}}

Return JSON only. Do not include explanations.
"""

In [11]:
import json

def parse_refinement_response(response: str, topics: dict, min_confidence: float = 0.0) -> list:
    """Parse merge operations from structured LLM JSON response."""
    
    if not response:
        return []

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        # fallback: extract JSON if wrapped in text
        start = response.find("{")
        end = response.rfind("}") + 1
        if start == -1 or end == -1:
            return []
        try:
            data = json.loads(response[start:end])
        except json.JSONDecodeError:
            return []

    merges = []

    for item in data.get("merges", []):
        merge_ids = item.get("merge_ids", [])
        kept_id = item.get("kept_id")
        confidence = float(item.get("confidence", 0.0))

        # validation
        if not merge_ids or kept_id not in merge_ids:
            continue

        if confidence < min_confidence:
            continue

        merges.append({
            "merge_ids": [int(x) for x in merge_ids],
            "kept_id": int(kept_id),
            "label": item.get("label", "").strip(),
            "description": item.get("description", "").strip(),
            "confidence": confidence
        })

    return merges


def refine_topics(topics: dict, subject: str, min_confidence: float = 0.7) -> dict:
    """Stage 2: Merge near-duplicate topics."""

    checkpoint = load_checkpoint("refinement", subject)
    if checkpoint is not None:
        if isinstance(checkpoint, dict) and "topics" in checkpoint:
            return checkpoint["topics"], checkpoint.get("old_to_new", {})
        return checkpoint, {}

    refined = dict(topics)

    topics_str = format_topics_for_prompt(refined)
    user_prompt = REFINEMENT_USER_TEMPLATE.format(topics=topics_str)

    response = call_llm(REFINEMENT_SYSTEM_PROMPT, user_prompt)

    merges = parse_refinement_response(response, refined, min_confidence=min_confidence)

    if not merges:
        print("  No merges needed")
    else:
        # Apply strongest merges first
        merges = sorted(merges, key=lambda x: x["confidence"], reverse=True)

        used_ids = set()

        print(f"  Applying {len(merges)} merge(s):")

        for m in merges:

            merge_ids = set(m["merge_ids"])
            kept_id = m["kept_id"]

            # Skip if topics already merged
            if merge_ids & used_ids:
                continue

            # Validate existence
            valid_ids = [i for i in merge_ids if i in refined]
            if len(valid_ids) < 2:
                continue

            print(
                f"    Merge {valid_ids} -> [{kept_id}] "
                f"{m['label']} (conf={m['confidence']:.2f})"
            )

            # Update kept topic
            if kept_id in refined:
                refined[kept_id] = {
                    "label": m["label"],
                    "description": m["description"]
                }

            # Remove merged topics
            for mid in valid_ids:
                if mid != kept_id and mid in refined:
                    del refined[mid]

            used_ids.update(valid_ids)

    # ---- Reindex topics sequentially ----

    reindexed = {}
    old_to_new = {}

    for new_id, (old_id, topic) in enumerate(sorted(refined.items()), start=1):
        reindexed[new_id] = topic
        old_to_new[old_id] = new_id

    save_checkpoint({"topics": reindexed, "old_to_new": old_to_new}, "refinement", subject)

    print(f"  Refined: {len(topics)} -> {len(reindexed)} topics")

    return reindexed, old_to_new

In [12]:
all_refined_topics = {}
all_refined_doc_maps = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC REFINEMENT: {subject.upper()}")
    print(f"{'='*60}")

    refined, old_to_new = refine_topics(all_topics[subject], subject)
    all_refined_topics[subject] = refined

    # Remap doc_map through merge operations
    raw_doc_map = all_doc_maps.get(subject, {})
    remapped = {doc_idx: old_to_new.get(tid, tid)
                for doc_idx, tid in raw_doc_map.items()
                if old_to_new.get(tid, tid) in refined}
    all_refined_doc_maps[subject] = remapped
    print(f"  doc_map: {len(raw_doc_map)} -> {len(remapped)} entries after remap")

    print(f"\n  Refined topics for {subject}:")
    for tid, t in sorted(refined.items()):
        print(f"    [{tid}] {t['label']}: {t['description']}")


TOPIC REFINEMENT: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/refinement.pkl
  doc_map: 8879 -> 8879 entries after remap

  Refined topics for cs:
    [1] Web Text Database Classification: Automated methods for categorizing and probing search-only text databases via query-based classification techniques
    [2] AI Story Understanding: Exploration of computational methods for interpreting narrative structures and semantic coherence in textual data, including historical shifts and agent-based approaches
    [3] Parallel Debugging and Extension Language Debugging Systems: Exploration of techniques for diagnosing errors in concurrent/parallel computing environments through visualization, nondeterminism handling, and distributed debugging mechanisms, with extensions to automated interactions across hardware/software layers.
    [4] Noninvertibility Theory: Exploration of mathematical and cryptographic properties defining invertibility limits in functions, particularly focusing o

In [13]:
show_significant_topics(all_refined_topics, all_refined_doc_maps, min_docs=2)


SIGNIFICANT TOPICS (Docs > 1): CS
ID    | Count    | Share %  | Topic Label
-----------------------------------------------------------------
14    | 1085     |   12.2%  | Repository Management Systems
20    | 498      |    5.6%  | Reproducibility Pitfalls
138   | 496      |    5.6%  | Periodic Numeration Systems
22    | 487      |    5.5%  | Dynamic Market Optimization
151   | 469      |    5.3%  | Time-Series Pattern Discovery
16    | 439      |    4.9%  | Polysemantic Feature Modeling and Ambiguity Handling in Logic Systems
141   | 292      |    3.3%  | Context-Dependent Classification
15    | 259      |    2.9%  | Recursive Definition Systems
17    | 255      |    2.9%  | Predictor-Based Parsing
24    | 222      |    2.5%  | Variable Word Rate Modeling
19    | 220      |    2.5%  | Foundations of Mathematical Randomness
71    | 211      |    2.4%  | Regulatory Agency Independence
27    | 205      |    2.3%  | Multi-Channel Rate Distortion Optimization
101   | 179      |    2.0%  |

---
## Stage 2.5: Topic Enrichment

Use the LLM to write a richer 2-3 sentence description per topic, informed by the actual documents assigned to it during generation.

In [14]:
ENRICHMENT_SYSTEM_PROMPT = """You are an expert academic topic analyst.
Given a topic name and representative paper abstracts, write a detailed 5-7 sentence description
capturing the topic's scope, key methods, and applications.

OUTPUT RULES:
1. Return ONLY valid JSON: {"enriched_description": "..."}
2. Use PLAIN TEXT only. No markdown, no bolding (**), and no bullet points (-).
3. If you use quotes inside the description, use 'single quotes' so the JSON doesn't break.
4. Keep the entire description on ONE SINGLE LINE. No newlines inside the JSON value."""

import json
import re

def clean_and_parse_json(response):
    text = re.sub(r"```json\s*|```", "", response).strip()
    
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1:
        return None
    
    json_str = text[start:end+1]
    
    json_str = json_str.replace('\n', ' ').replace('\r', '')
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            # Matches everything between "enriched_description": " and the final "
            match = re.search(r'"enriched_description":\s*"(.*)"', json_str)
            if match:
                content = match.group(1)
                return {"enriched_description": content}
        except:
            pass
    return None

def enrich_topics(refined_topics: dict, doc_map: dict, texts: list, subject: str, top_k: int = 8) -> dict:
    """Stage 2.5: enriched descriptions via LLM using assigned docs."""
    checkpoint = load_checkpoint("enrichment", subject)
    if checkpoint is not None:
        return checkpoint["enriched_topics"]

    from collections import defaultdict
    topic_docs = defaultdict(list)
    for doc_idx, tid in doc_map.items():
        if tid in refined_topics:
            topic_docs[tid].append(doc_idx)

    enriched = {}
    for tid, topic in tqdm(refined_topics.items(), desc=f"Enriching {subject}"):
        all_indices = topic_docs.get(tid, [])
        step = max(1, len(all_indices) // top_k)
        doc_indices = all_indices[::step][:top_k] # Takes a representative spread

        # print(f"All indiecies len {len(all_indices)}, after take represeitative : {len(doc_indices)}")
        snippets = [texts[i][:2000] for i in doc_indices if i < len(texts)]
        # print(f"Snippets count : {len(snippets)}\n")
        snippets_str = "\n\n".join(f"Abstract {i+1}:\n{s}" for i, s in enumerate(snippets)) or "(no docs assigned)"
        # user_prompt = (
        #     f"Topic: {topic['label']}\n"
        #     f"Current description: {topic['description']}\n\n"
        #     f"Representative abstracts:\n{snippets_str}\n\n"
        #     "Write an enriched description as JSON."
        # )
        user_prompt = (
    f"Topic Label: {topic['label']}\n"
    f"Initial Definition: {topic['description']}\n\n" # Help it stay on track
    f"New Evidence (Abstracts):\n{snippets_str}\n\n"
    "Task: Synthesize the Initial Definition with the New Evidence to create a "
    "comprehensive, technical description. If the abstracts provide more specific "
    "methods or applications than the initial definition, prioritize the abstracts."
 )

        response = call_llm(ENRICHMENT_SYSTEM_PROMPT, user_prompt)

        parsed_data = clean_and_parse_json(response)

        enriched_desc = topic.get("description", "No description available.")
        if parsed_data and "enriched_description" in parsed_data:
            enriched_desc = parsed_data["enriched_description"]
        else:
            print(f"  [Warning] Parse failed for Topic {tid}, using fallback.")

        enriched[tid] = {**topic, "enriched_description": enriched_desc}
        if len(enriched) % 50 == 0:
            save_checkpoint({"enriched_topics": enriched}, "enrichment", subject)
        # print(f"Enriched: {enriched[tid]}")

    save_checkpoint({"enriched_topics": enriched}, "enrichment", subject)
    return enriched

In [15]:
all_enriched_topics = {}
for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC ENRICHMENT: {subject.upper()}")
    print(f"{'='*60}")
    texts_subj = all_data[subject]["text"].fillna("").tolist()
    enriched = enrich_topics(
        all_refined_topics[subject],
        all_refined_doc_maps[subject],
        texts_subj, subject,
        top_k=20
    )
    all_enriched_topics[subject] = enriched
    print(f"  Enriched {len(enriched)} topics")


TOPIC ENRICHMENT: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/enrichment.pkl
  Enriched 616 topics

TOPIC ENRICHMENT: MATH
  Checkpoint loaded: ../../../../models/topicGpt/math/enrichment.pkl
  Enriched 610 topics

TOPIC ENRICHMENT: PHYSICS
  Checkpoint loaded: ../../../../models/topicGpt/physics/enrichment.pkl
  Enriched 504 topics


---
## Stage 3: Topic Assignment (Sentence Transformers)

Assign every document to the closest enriched topic using cosine similarity over pre-computed `.mmap` embeddings.
A tuning loop evaluates all 6 available embedding models and selects the best by Topic Quality (harmonic mean of C_v coherence and IRBO diversity).

In [ ]:
from pathlib import Path
import numpy as np

EMBEDDING_DIR = Path("../../../../embedding")

MODEL_HF_MAP = {
    "BAAI_bge_base_en_v1.5":                    "BAAI/bge-base-en-v1.5",
    "all_distilroberta_v1":                       "sentence-transformers/all-distilroberta-v1",
    "all_mpnet_base_v2":                          "sentence-transformers/all-mpnet-base-v2",
    # "allenai_specter2":                           "allenai/specter2",
    "intfloat_e5_base_v2":                        "intfloat/e5-base-v2",
    "sentence_transformers_all_MiniLM_L6_v2":     "sentence-transformers/all-MiniLM-L6-v2",
}

import re

def list_embedding_models(subject: str):
    return sorted(re.sub(r'_v1$', '', f.stem) for f in (EMBEDDING_DIR / subject).glob("*.mmap"))

def load_doc_embeddings(subject: str, model_name: str) -> np.ndarray:
    meta_path = EMBEDDING_DIR / subject / f"{model_name}_meta_v1.npy"
    if not meta_path.exists():
        meta_path = EMBEDDING_DIR / subject / f"{model_name}_v1_meta.npy"
    meta  = np.load(meta_path, allow_pickle=True).item()
    n, d  = meta["n_samples"], meta["emb_dim"]
    return np.memmap(
        EMBEDDING_DIR / subject / f"{model_name}_v1.mmap",
        dtype="float32", mode="r", shape=(n, d)
    )


In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

def assign_topics_st(doc_embs, enriched_topics: dict, model_name: str, batch_size=2000, threshold=0.5) -> pd.DataFrame:
    """Cosine-similarity assignment using a SentenceTransformer model with outlier thresholding."""
    hf_name    = MODEL_HF_MAP.get(model_name, model_name)
    st_model   = SentenceTransformer(hf_name)
    topic_ids  = sorted(enriched_topics.keys())
    
    descs      = [enriched_topics[t].get("enriched_description") or enriched_topics[t]["description"]
                  for t in topic_ids]
    
    topic_embs = st_model.encode(descs, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    
    rows = []
    for start in tqdm(range(0, len(doc_embs), batch_size), desc=f"Assigning ({model_name})", leave=False):
        batch = np.array(doc_embs[start:start+batch_size])
        
        sims  = cos_sim(batch, topic_embs)
        best  = sims.argmax(axis=1)
        scores = sims.max(axis=1)
        
        for i, (bi, sc) in enumerate(zip(best, scores)):
            if sc >= threshold:
                tid = topic_ids[bi]
                label = enriched_topics[tid]["label"]
            else:
                tid = -1
                label = "Outlier / Unassigned"
                
            rows.append({
                "doc_idx": start + i, 
                "topic_id": tid,
                "topic_label": label,
                "confidence": float(sc)
            })
    print(f"Len rows = {len(rows)}")       
    return pd.DataFrame(rows)

def compute_topic_words_ctfidf(assignment_df: pd.DataFrame, texts: list, top_n: int = 10) -> dict:
    from sklearn.feature_extraction.text import TfidfVectorizer
    import pandas as pd

    topic_docs = []
    tids = []
    
    for tid, grp in assignment_df.groupby("topic_id"):
        if tid == -1: continue 
        
        combined_text = " ".join([texts[i] for i in grp["doc_idx"].tolist() if i < len(texts)])
        topic_docs.append(combined_text)
        tids.append(tid)

    if not topic_docs:
        return {}

    vec = TfidfVectorizer(
        stop_words="english", 
        max_features=10000,
        ngram_range=(1, 2) 
    )
    
    tfidf_matrix = vec.fit_transform(topic_docs)
    feature_names = vec.get_feature_names_out()
    
    topic_words = {}
    for i, tid in enumerate(tids):
        row = tfidf_matrix.getrow(i).toarray().flatten()
        top_ids = row.argsort()[-top_n:][::-1]
        topic_words[tid] = [feature_names[idx] for idx in top_ids if row[idx] > 0]
    print(topic_words)

    return topic_words


def compute_coherence_irbo(assignment_df: pd.DataFrame, texts: list, top_n: int = 10) -> dict:
    """Compute C_v coherence and IRBO diversity, return dict with both + TQ."""
    topic_words = compute_topic_words_ctfidf(assignment_df, texts, top_n)
    word_lists  = [v for v in topic_words.values() if v]
    tokenized   = [t.lower().split() for t in texts]

    try:
        dct = Dictionary(tokenized)
        cm  = CoherenceModel(topics=word_lists, texts=tokenized, dictionary=dct, coherence="c_v", processes=1)
        cv  = cm.get_coherence()
    except Exception as e:
        print(f"    Coherence error: {e}")
        cv = 0.0

    def rbo(l1, l2, p=RBO_P):
        score, weight, s1, s2 = 0.0, 1.0, set(), set()
        for d in range(1, min(len(l1), len(l2)) + 1):
            s1.add(l1[d-1]); s2.add(l2[d-1])
            score += weight * len(s1 & s2) / d
            weight *= p
        return 1 - score  # inverse = diversity

    pairs = list(combinations(word_lists, 2))
    irbo  = float(np.mean([rbo(a, b) for a, b in pairs])) if pairs else 0.0
    tq    = 2 * cv * irbo / (cv + irbo + 1e-8)
    return {"coherence": cv, "irbo": irbo, "topic_quality": tq}

In [23]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
all_results = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TUNING: {subject.upper()}")
    print(f"{'='*60}")
    texts_subj = all_data[subject]["text"].fillna("").tolist()
    models     = list_embedding_models(subject)
    best_score, best_res = -1, None

    for model_name in models:
        print(f"  Model: {model_name}")
    
        if model_name == "allenai_specter2":
            continue
        ckpt = load_checkpoint(f"assignment_{model_name}", subject)
        if ckpt:
            df_assign = ckpt["assignment_df"]
            metrics   = ckpt["metrics"]
        else:
            doc_embs  = load_doc_embeddings(subject, model_name)
            df_assign = assign_topics_st(doc_embs, all_enriched_topics[subject], model_name)
            metrics   = compute_coherence_irbo(df_assign, texts_subj)
            save_checkpoint({"assignment_df": df_assign, "metrics": metrics},
                            f"assignment_{model_name}", subject)

        print(f"    C_v={metrics['coherence']:.4f}  IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}")
        if metrics["topic_quality"] > best_score:
            best_score = metrics["topic_quality"]
            best_res   = {"model": model_name, "df": df_assign, "metrics": metrics}

    all_results[subject] = best_res
    print(f"  Best: {best_res['model']} (TQ={best_score:.4f})")

# Save CSV results
for subject, res in all_results.items():
    df = res["df"].copy()
    df["subject"]    = subject
    df["best_model"] = res["model"]
    df["coherence"]  = res["metrics"]["coherence"]
    df["irbo"]       = res["metrics"]["irbo"]
    out = RESULT_DIR / subject / "topicgpt_assignments.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)
    print(f"Saved: {out}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Rows: {len(df)}")


TUNING: CS
  Model: BAAI_bge_base_en_v1.5
  Checkpoint loaded: ../../../../models/topicGpt/cs/assignment_BAAI_bge_base_en_v1.5.pkl
    C_v=0.5814  IRBO=0.8211  TQ=0.6808
  Model: all_distilroberta_v1
  Checkpoint loaded: ../../../../models/topicGpt/cs/assignment_all_distilroberta_v1.pkl
    C_v=0.6034  IRBO=0.9046  TQ=0.7239
  Model: all_mpnet_base_v2
  Checkpoint loaded: ../../../../models/topicGpt/cs/assignment_all_mpnet_base_v2.pkl
    C_v=0.6007  IRBO=0.8897  TQ=0.7172
  Model: allenai_specter2
  Model: intfloat_e5_base_v2


Len rows = 165756
{1: ['text', 'web', 'classification', 'document', 'classifier', 'search', 'databases', 'data mining', 'data', 'approach'], 2: ['story', 'narrative', 'stories', 'narratives', 'social', 'data', 'understanding', 'language', 'affective', 'textual'], 3: ['debugging', 'mpi', 'programs', 'program', 'bugs', 'based', 'programming', 'fault', 'language', 'code'], 4: ['quantum', 'security', 'cryptographic', 'encryption', 'protocol', 'cryptography', 'public key', 'based', 'key', 'secure'], 5: ['logic', 'reasoning', 'default', 'logics', 'semantics', 'logic programs', 'monotonic', 'non', 'propositional', 'encodings'], 6: ['fuzzy', 'neuro', 'soft', 'actuator', 'control', 'controller', 'torque', 'neural', 'model', 'based'], 7: ['retrieval', 'documents', 'based', 'rag', 'search', 'document', 'query', 'ranking', 'model', 'information'], 8: ['type', 'types', 'typed', 'lambda', 'typing', 'programming', 'languages', 'inference', 'intersection', 'calculus'], 9: ['probabilistic', 'logic', 'p

Len rows = 165756
{1: ['search', 'documents', 'web', 'query', 'document', 'text', 'information', 'based', 'data', 'retrieval'], 2: ['dialogue', 'models', 'sentiment', 'language', 'text', 'story', 'conversational', 'model', 'dataset', 'social'], 3: ['debugging', 'mpi', 'program', 'fault', 'programs', 'bugs', 'execution', 'code', 'based', 'software'], 4: ['finite fields', 'quantum', 'bit', 'cryptography', 'coq', 'cryptographic', 'finite', 'smt', 'procedure', 'functions'], 5: ['reasoning', 'logic', 'datalog', 'query', 'knowledge', 'language', 'llms', 'logical', 'models', 'logic programming'], 6: ['fuzzy', 'neural', 'reservoir', 'neuro', 'neural network', 'learning', 'network', 'model', 'control', 'systems'], 7: ['retrieval', 'query', 'relevance', 'documents', 'document', 'ranking', 'search', 'ir', 'models', 'information retrieval'], 8: ['type', 'subtyping', 'type theory', 'typed', 'types', 'typing', 'lambda', 'theory', 'programming', 'languages'], 9: ['probabilistic', 'inference', 'logic'

Len rows = 157085
{1: ['adic', 'mathbb', 'groups', 'archimedean', 'non archimedean', 'uniformization', 'group', 'arithmetic', 'pp adic', 'field'], 2: ['laplacian', 'eigenvalue', 'eigenvalues', 'riemannian', 'spectrum', 'spectral', 'manifolds', 'isospectral', 'curvature', 'operator'], 3: ['banach', 'spaces', 'banach spaces', 'space', 'banach space', 'ell', 'mathbb', 'mathcal', 'operators', 'lipschitz'], 4: ['mathcal', 'mathbb', 'vector', 'spaces', 'linear', 'vector spaces', 'graded', 'space', 'algebra', 'mathcal mathcal'], 5: ['bayesian', 'posterior', 'inference', 'likelihood', 'bayes', 'prior', 'estimation', 'data', 'priors', 'frequentist'], 6: ['finite element', 'equations', 'homogenization', 'method', 'problem', 'solutions', 'boundary', 'equation', 'stokes', 'numerical'], 7: ['operators', 'mathbb', 'mathcal', 'spaces', 'space', 'operator', 'bergman', 'hilbert', 'kernel', 'functions'], 8: ['mirror', 'calabi yau', 'calabi', 'yau', 'mirror symmetry', 'cohomology', 'toric', 'varieties', 

Len rows = 157085
{1: ['mathbb', 'adic', 'mathbb mathbb', 'arithmetic', 'field', 'galois', 'pp adic', 'pp', 'groups', 'group'], 2: ['laplacian', 'eigenvalue', 'isospectral', 'manifolds', 'riemannian', 'spectrum', 'eigenvalues', 'curvature', 'laplace', 'metrics'], 3: ['banach', 'spaces', 'mathcal', 'banach spaces', 'space', 'mathbb', 'banach space', 'operators', 'infty', 'ell'], 4: ['mathbb', 'mathfrak', 'polynomials', 'polynomial', 'graded', 'algebras', 'mathbb mathbb', 'algebra', 'gr bner', 'bner'], 5: ['bayesian', 'posterior', 'inference', 'bayes', 'gaussian', 'prior', 'likelihood', 'data', 'variational', 'priors'], 6: ['homogenization', 'homogenized', 'varepsilon', 'periodic', 'scale', 'equations', 'heterogeneous', 'media', 'finite element', 'diffusion'], 7: ['operators', 'operator', 'mathcal', 'hilbert', 'kernels', 'space', 'mathbb', 'spaces', 'krein', 'self adjoint'], 8: ['calabi yau', 'mirror', 'calabi', 'yau', 'mathbb', 'varieties', 'toric', 'mirror symmetry', 'cohomology', 'fla

...

Len rows = 146311
{1: ['magnetic', 'field', 'electron', 'quantum', 'spin', 'magnetic field', 'relativistic', 'atoms', 'electric', 'hyperfine'], 2: ['risk', 'air', 'flight', 'aircraft', 'data', 'emissions', 'traffic', 'fuel', 'environmental', 'dust'], 3: ['magnetic', 'field', 'fracture', 'high', 'model', 'current', 'coil', 'hts', 'crack', 'magnets'], 4: ['beam', 'muon', 'plasma', 'beams', 'electron', 'emittance', 'wakefield', 'colliders', 'accelerator', 'energy'], 5: ['uncertainty', 'bayesian', 'data', 'model', 'uncertainty quantification', 'method', 'posterior', 'uncertainties', 'inference', 'based'], 6: ['muonic', 'corrections', 'relativistic', 'lamb shift', 'nuclear', 'qed', 'hyperfine', 'lamb', 'electron', 'ions'], 7: ['relativity', 'gravitational', 'general relativity', 'gravity', 'newtonian', 'theory', 'spacetime', 'general', 'gravitation', 'metric'], 8: ['dna', 'model', 'protein', 'chromatin', 'proteins', 'conformational', 'coarse grained', 'dynamics', 'molecular', 'coarse'], 9: 

Len rows = 146311
{1: ['quantum', 'electronic', 'field', 'magnetic', 'mathcal pt', 'spin', 'electron', 'basis', 'atoms', 'electronic structure'], 2: ['weather', 'meteorological', 'flight', 'air', 'routing', 'co2', 'emissions', 'risk', 'data', 'safety'], 3: ['wires', 'wire', 'magnets', 'nb', 'superconductors', 'reconnection', 'guide field', 'coils', 'current', 'electrodes'], 4: ['plasma', 'beam', 'plasmas', 'electron', 'wakefield', 'magnetic', 'laser', 'particle', 'ion', 'field'], 5: ['uncertainty', 'bayesian', 'data', 'posterior', 'model', 'inference', 'uncertainty quantification', 'inverse problems', 'quantification', 'learning'], 6: ['muonic', 'relativistic', 'corrections', 'lamb shift', 'hydrogen', 'hyperfine', 'nuclear', 'qed', 'muonic hydrogen', 'lamb'], 7: ['gravitational', 'relativity', 'gravity', 'general relativity', 'newtonian', 'einstein', 'gravitation', 'spacetime', 'theory', 'metric'], 8: ['dna', 'protein', 'folding', 'proteins', 'model', 'molecular', 'chromatin', 'dynamic

Len rows = 146311
{1: ['quantum', 'hamiltonian', 'magnetic', 'field', 'spin', 'electronic', 'electron', 'vqe', 'state', 'systems'], 2: ['air', 'data', 'risk', 'traffic', 'aerosol', 'weather', 'plume', 'health', 'model', 'forecasting'], 3: ['wire', 'current', 'wires', 'magnets', 'magnetic', 'field', 'tearing', 'breakdown', 'high', 'runaway'], 4: ['plasma', 'beam', 'electron', 'plasmas', 'ion', 'particle', 'magnetic', 'particle cell', 'field', 'simulations'], 5: ['bayesian', 'uncertainty', 'posterior', 'inference', 'data', 'model', 'uncertainty quantification', 'uncertainties', 'monte', 'monte carlo'], 6: ['nuclear', 'muonic', 'relativistic', 'hyperfine', 'electron', 'corrections', 'calculations', 'ions', 'hydrogen', 'atomic'], 7: ['relativity', 'newtonian', 'gravitational', 'general relativity', 'gravity', 'motion', 'newton', 'general', 'gravitation', 'kepler'], 8: ['dna', 'chromatin', 'protein', 'model', 'proteins', 'molecular', 'rna', 'dynamics', 'stranded', 'transcription'], 9: ['dee

Len rows = 146311
{1: ['field', 'magnetic', 'adjoint', 'fields', 'magnetic fields', 'atoms', 'equation', 'strong', 'equations', 'electric'], 2: ['aircraft', 'air', 'emissions', 'traffic', 'aerosol', 'models', 'weather', 'fatigue', 'data', 'meteorological'], 3: ['wire', 'wires', 'high', 'current', 'voltage', 'field', 'breakdown', 'magnetic', 'coil', 'superconducting'], 4: ['plasma', 'beam', 'electron', 'beams', 'ion', 'particle', 'rf', 'plasma density', 'particle cell', 'sources'], 5: ['bayesian', 'model', 'uncertainty', 'data', 'inference', 'models', 'method', 'based', 'posterior', 'learning'], 6: ['muonic', 'lamb shift', 'lamb', 'corrections', 'hydrogen', 'hyperfine', '2p', 'nuclear', 'hyperfine splitting', 'shift'], 7: ['newtonian', 'relativity', 'gravitational', 'space', 'motion', 'gravity', 'geodesic', 'general relativity', 'theory', 'geometry'], 8: ['dna', 'model', 'protein', 'proteins', 'chromatin', 'conformational', 'coarse grained', 'dynamics', 'grained', 'coarse'], 9: ['data',

Len rows = 146311
{1: ['quantum', 'electron', 'magnetic', 'hamiltonian', 'electronic', 'method', 'field', 'spin', 'energy', 'states'], 2: ['risk', 'combustion', 'weather', 'learning', 'ml', 'data', 'machine', 'machine learning', 'forecasting', 'routing'], 3: ['wire', 'wires', 'reconnection', 'plasma', 'sheets', 'array', 'tearing', 'current sheets', 'ablation', 'resistive'], 4: ['plasma', 'beam', 'electron', 'wakefield', 'bunch', 'beams', 'laser', 'ion', 'accelerators', 'particle'], 5: ['uncertainty', 'bayesian', 'uncertainty quantification', 'model', 'quantification', 'data', 'posterior', 'inverse problems', 'inference', 'uncertainties'], 6: ['muonic', 'hydrogen', 'relativistic', 'corrections', 'muonic hydrogen', 'lamb shift', 'lamb', 'hyperfine', 'nuclear', 'energy'], 7: ['relativity', 'general relativity', 'gravity', 'newtonian', 'gravitational', 'metric', 'spacetime', 'geodesic', 'general', 'motion'], 8: ['dna', 'protein', 'folding', 'proteins', 'conformational', 'model', 'molecular

In [24]:
def print_final_summary(all_results, all_enriched_topics):
    print(f"\n{'='*60}")
    print(f"{'SUBJECT':<12} | {'TOPICS':<8} | {'DOCS':<8} | {'BEST MODEL'}")
    print(f"{'-'*60}")
    
    for subject, res in all_results.items():
        df = res["df"]
        # Count unique topics (excluding outlier -1 if present)
        n_topics = len(all_enriched_topics[subject])
        n_docs = len(df)
        model = res["model"]
        
        print(f"{subject:<12} | {n_topics:<8} | {n_docs:<8} | {model}")

# Run the summary
print_final_summary(all_results, all_enriched_topics)


SUBJECT      | TOPICS   | DOCS     | BEST MODEL
------------------------------------------------------------
cs           | 616      | 165756   | all_distilroberta_v1
math         | 610      | 157085   | sentence_transformers_all_MiniLM_L6_v2
physics      | 504      | 146311   | sentence_transformers_all_MiniLM_L6_v2


In [25]:
for subject, res in all_results.items():
    print(f"\nDetailed Counts for {subject.upper()} (Model: {res['model']}):")
    df = res["df"]
    
    # Get counts of every topic_id
    counts = df["topic_id"].value_counts().sort_index()
    
    # Map the labels back for readability
    for tid, count in counts.items():
        if tid == -1:
            label = "OUTLIER"
        else:
            label = all_enriched_topics[subject][tid]['label']
            
        print(f"  [{tid:>3}] {label:<30} : {count} docs")


Detailed Counts for CS (Model: all_distilroberta_v1):
  [ -1] OUTLIER                        : 90774 docs
  [  1] Web Text Database Classification : 211 docs
  [  2] AI Story Understanding         : 129 docs
  [  3] Parallel Debugging and Extension Language Debugging Systems : 179 docs
  [  4] Noninvertibility Theory        : 17 docs
  [  5] Nonmonotonic Logic Encoding    : 62 docs
  [  6] Neuro-Fuzzy Control Systems    : 57 docs
  [  7] Novelty-Based Retrieval Evaluation : 369 docs
  [  8] Type Class Extensions          : 372 docs
  [  9] Quantitative Probabilistic Logic Programming : 93 docs
  [ 10] Prosody-Based Segmentation     : 230 docs
  [ 11] Dynamic Semantics Resolution   : 121 docs
  [ 12] Repair-Based Speech Correction : 134 docs
  [ 13] Object-Oriented Music Systems  : 48 docs
  [ 14] Repository Management Systems  : 6 docs
  [ 15] Recursive Definition Systems   : 50 docs
  [ 16] Polysemantic Feature Modeling and Ambiguity Handling in Logic Systems : 210 docs
  [ 17] Predi